In [7]:
is_colab = 'google.colab' in str(get_ipython())


## Installing and importing nessisary dependencies

In [8]:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# %pip install gymnasium
# %pip install 'stable-baselines3[extra]'

In [9]:
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO
import gymnasium as gym
import os

# Colab Imports
if is_colab:
    %pip install gymnasium stable-baselines3[extra] pygame
    %pip install pyvirtualdisplay
    !apt-get install -y xvfb

    from pyvirtualdisplay import Display
    from IPython.display import Image
    import imageio

    display = Display(visible=0, size=(400, 300))
    display.start()


## Create enviroment for the model

In [ ]:
enviroment_name = 'CartPole-v1'
env = gym.make(id=enviroment_name)

: 

## Test algorithm

In [ ]:
episodes = 100

for episode in range(1, episodes+1):
    state, _ = env.reset()
    done = False
    score = 0

    while not done:
        if is_colab:
            frames.append(env.render())
        action = env.action_space.sample()
        n_state, reward, done, info, _ = env.step(action=action)
        score += reward
    print(f'Episode:{episode} Score:{score}')
env.close()
if is_colab:
    imageio.mimsave('random_agent.gif', frames, fps=30)
    Image('random_agent.gif')

## Training the model

In [3]:
log_path = os.path.join(os.getcwd(), "Training", "Logs")
model_path = os.path.join(os.getcwd(), "Training", "Models")

In [ ]:
env = gym.make(enviroment_name)  # no render_mode
env = DummyVecEnv([lambda: env])
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=log_path)

Using cpu device


In [ ]:
model.learn(total_timesteps=20000)

## Save and Reload model

In [7]:
PPO_path = os.path.join(model_path, 'PPO_Model_CartPole')

In [ ]:
model.save(PPO_path)

In [5]:
del model

In [8]:
model = PPO.load(PPO_path, env)

## Evaluation

In [ ]:
evaluate_policy(model, env, n_eval_episodes=10)

In [ ]:
episodes = 500

if is_colab:
    eval_env = gym.make('CartPole-v1', render_mode='rgb_array')
    frames = []

    obs, _ = eval_env.reset()
    for _ in range(episodes):
        action, _ = model.predict(obs)
        obs, reward, done, _, info = eval_env.step(action)
        frames.append(eval_env.render())
        if done:
            obs, _ = eval_env.reset()

    eval_env.close()
    imageio.mimsave('trained_agent.gif', frames, fps=30)
    Image('trained_agent.gif')
else:
    eval_env = gym.make('CartPole-v1')
    obs, _ = eval_env.reset()
    for _ in range(episodes):
        action, _ = model.predict(obs)
        obs, reward, done, _, info = eval_env.step(action)
        if done:
            obs, _ = eval_env.reset()
    eval_env.close() 